# Module 05: Scikit-Learn for Machine Learning
## Notebook 01: Preprocessing and Production Pipeline Design

Scikit-Learn is the gold standard machine learning framework in Python. Its unified, elegant API architecture (`fit`, `transform`, `predict`) forms the bedrock of production machine learning systems.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Master Scikit-Learn's core interface conventions: **Estimators**, **Transformers**, and **Predictors**.
2. Partition datasets using `train_test_split()` with class stratification to prevent distribution shifts.
3. Scale continuous features using `StandardScaler`, `MinMaxScaler`, and `RobustScaler`.
4. Encode categorical variables using `OneHotEncoder(handle_unknown='ignore')`.
5. Combine mixed-type transformations using **`ColumnTransformer`**.
6. Assemble leak-free, production-ready machine learning **`Pipeline`** workflows.

In [1]:
import sklearn
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

print(f"Scikit-Learn version: {sklearn.__version__}")

Scikit-Learn version: 1.9.1


### 1. The Core Scikit-Learn API Pattern

Scikit-Learn organizes all functionality around three object interfaces:
1. **Estimator**: Any object that learns parameters from data via `.fit(X, y)`.
2. **Transformer**: An estimator that transforms data via `.transform(X)`.
3. **Predictor**: An estimator that makes predictions on unseen data via `.predict(X)` or `.predict_proba(X)`.

> **The Golden Rule: Never Leak Test Information**
> - Call `.fit()` or `.fit_transform()` **ONLY** on the Training Set!
> - Call `.transform()` on the Validation and Test Sets.

In [2]:
# Simulated raw dataset with numerical, categorical, and missing features
raw_data = {
    'Age': [25.0, 42.0, np.nan, 35.0, 58.0, 29.0, 48.0, 31.0],
    'Salary': [50000.0, 85000.0, 62000.0, np.nan, 120000.0, 48000.0, 95000.0, 72000.0],
    'Department': ['Sales', 'Engineering', 'HR', 'Engineering', 'Sales', 'HR', 'Engineering', 'Sales'],
    'Purchased': [0, 1, 0, 1, 1, 0, 1, 0]
}

df = pd.DataFrame(raw_data)
X = df.drop(columns=['Purchased'])
y = df['Purchased']

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print("Training feature samples:\n", X_train)

X_train shape: (6, 3) | X_test shape: (2, 3)
Training feature samples:
     Age   Salary   Department
1  42.0  85000.0  Engineering
2   NaN  62000.0           HR
3  35.0      NaN  Engineering
6  48.0  95000.0  Engineering
0  25.0  50000.0        Sales
7  31.0  72000.0        Sales


---
### 2. Building Sub-Pipelines for Numerical and Categorical Features

Real-world datasets contain different types of columns:
- **Numerical columns** need missing value imputation (median) followed by feature scaling (`StandardScaler`).
- **Categorical columns** need missing value imputation (most frequent) followed by One-Hot Encoding (`handle_unknown='ignore'`).

In [3]:
num_features = ['Age', 'Salary']
cat_features = ['Department']

# 1. Pipeline for numerical features
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Pipeline for categorical features
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

---
### 3. Assembling with `ColumnTransformer` and Full `Pipeline`

`ColumnTransformer` applies each specific sub-pipeline to its designated list of columns and concatenates the resulting numerical feature arrays side-by-side.

In [4]:
# 3. Combine with ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

# 4. Create Full End-to-End Pipeline (Preprocessing + Estimator)
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Fit full pipeline on training set
full_pipeline.fit(X_train, y_train)

# Predict on test set directly from raw DataFrame!
test_preds = full_pipeline.predict(X_test)
test_probs = full_pipeline.predict_proba(X_test)

print("Test Predictions:    ", test_preds)
print("Test Probabilities:\n", np.round(test_probs, 4))

Test Predictions:     [0 1]
Test Probabilities:
 [[0.8808 0.1192]
 [0.0377 0.9623]]


### Summary & Next Steps
In this notebook, you mastered:
- Scikit-Learn's Estimator, Transformer, and Predictor architecture.
- Stratified data partitioning with `train_test_split`.
- Handling numeric and categorical data with `ColumnTransformer`.
- Assembling leak-free, production-grade end-to-end `Pipeline` workflows.

**Next Notebook:** `02_supervised_regression_models.ipynb` — Ordinary Least Squares, Ridge ($L_2$), Lasso ($L_1$), residual analysis, and regression evaluation metrics.